# AI Trading Agent with Multiple Mathematical Models

This notebook builds a trading framework using several mathematical models beyond Markov chains. It loads real data from the `data/SP500.csv` file and generates buy/sell signals with:

- Exponential Moving Average crossover
- RSI overbought/oversold
- Mean reversion via z-score
- Momentum
- Simple Kalman filter
- ARIMA forecasting

In [21]:
import math
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import pandas as pd

In [22]:
@dataclass
class Signal:
    timestamp: pd.Timestamp
    model_name: str
    signal: str
    score: float


class DataLoader:
    @staticmethod
    def load_csv(path: str, price_column: Optional[str] = None) -> pd.DataFrame:
        df = pd.read_csv(path)

        if "Date" in df.columns:
            df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
            df = df.dropna(subset=["Date"])
            df = df.sort_values("Date").reset_index(drop=True)
            df = df.rename(columns={"Date": "timestamp"})
        else:
            df = df.reset_index().rename(columns={"index": "timestamp"})

        if price_column is None:
            if "Close" in df.columns:
                price_column = "Close"
            elif "Price" in df.columns:
                price_column = "Price"
            else:
                price_column = df.columns[1]

        df = df[["timestamp", price_column]].rename(columns={price_column: "price"})
        df["price"] = pd.to_numeric(df["price"], errors="coerce")
        return df.dropna().reset_index(drop=True)


class FeatureCalculator:
    @staticmethod
    def exponential_moving_average(values: np.ndarray, window: int) -> np.ndarray:
        alpha = 2 / (window + 1)
        ema = np.empty_like(values)
        ema[0] = values[0]
        for t in range(1, len(values)):
            ema[t] = alpha * values[t] + (1 - alpha) * ema[t - 1]
        return ema

    @staticmethod
    def relative_strength_index(values: np.ndarray, window: int) -> np.ndarray:
        deltas = np.diff(values)
        gains = np.maximum(deltas, 0)
        losses = np.maximum(-deltas, 0)
        avg_gain = np.zeros_like(values)
        avg_loss = np.zeros_like(values)
        avg_gain[window] = gains[:window].mean()
        avg_loss[window] = losses[:window].mean()
        for i in range(window + 1, len(values)):
            avg_gain[i] = (avg_gain[i - 1] * (window - 1) + gains[i - 1]) / window
            avg_loss[i] = (avg_loss[i - 1] * (window - 1) + losses[i - 1]) / window
        rs = np.divide(avg_gain, avg_loss, out=np.zeros_like(avg_gain), where=avg_loss != 0)
        rsi = 100 - 100 / (1 + rs)
        rsi[: window + 1] = np.nan
        return rsi

    @staticmethod
    def zscore(values: np.ndarray, window: int) -> np.ndarray:
        zscores = np.full_like(values, np.nan)
        for i in range(window - 1, len(values)):
            window_values = values[i - window + 1 : i + 1]
            mean = window_values.mean()
            std = window_values.std(ddof=0)
            zscores[i] = (values[i] - mean) / (std if std > 0 else 1)
        return zscores

In [23]:
class StrategyModel:
    def __init__(self, name: str, warmup: int = 1) -> None:
        self.name = name
        self.warmup = warmup

    def update(self, data: pd.DataFrame) -> Optional[Signal]:
        raise NotImplementedError


class MovingAverageCrossoverModel(StrategyModel):
    def __init__(self, fast_window: int = 8, slow_window: int = 21) -> None:
        super().__init__(name="MovingAverageCrossover", warmup=slow_window)
        self.fast_window = fast_window
        self.slow_window = slow_window

    def update(self, data: pd.DataFrame) -> Optional[Signal]:
        prices = data["price"].to_numpy()
        if len(prices) < self.slow_window:
            return None

        fast = FeatureCalculator.exponential_moving_average(prices, self.fast_window)
        slow = FeatureCalculator.exponential_moving_average(prices, self.slow_window)
        if len(fast) < 2 or len(slow) < 2:
            return None

        if fast[-1] > slow[-1] and fast[-2] <= slow[-2]:
            return Signal(data["timestamp"].iloc[-1], self.name, "buy", float(fast[-1] - slow[-1]))
        if fast[-1] < slow[-1] and fast[-2] >= slow[-2]:
            return Signal(data["timestamp"].iloc[-1], self.name, "sell", float(slow[-1] - fast[-1]))
        return None


class RSIModel(StrategyModel):
    def __init__(self, window: int = 14, oversold: int = 30, overbought: int = 70) -> None:
        super().__init__(name="RSI", warmup=window + 1)
        self.window = window
        self.oversold = oversold
        self.overbought = overbought

    def update(self, data: pd.DataFrame) -> Optional[Signal]:
        prices = data["price"].to_numpy()
        if len(prices) < self.window + 1:
            return None

        rsi = FeatureCalculator.relative_strength_index(prices, self.window)
        if np.isnan(rsi[-1]):
            return None

        if rsi[-1] < self.oversold:
            return Signal(data["timestamp"].iloc[-1], self.name, "buy", float(self.oversold - rsi[-1]))
        if rsi[-1] > self.overbought:
            return Signal(data["timestamp"].iloc[-1], self.name, "sell", float(rsi[-1] - self.overbought))
        return None


class MeanReversionModel(StrategyModel):
    def __init__(self, window: int = 20, threshold: float = 2.0) -> None:
        super().__init__(name="MeanReversion", warmup=window)
        self.window = window
        self.threshold = threshold

    def update(self, data: pd.DataFrame) -> Optional[Signal]:
        prices = data["price"].to_numpy()
        if len(prices) < self.window:
            return None

        zscores = FeatureCalculator.zscore(prices, self.window)
        z = zscores[-1]
        if np.isnan(z):
            return None

        if z > self.threshold:
            return Signal(data["timestamp"].iloc[-1], self.name, "sell", float(z))
        if z < -self.threshold:
            return Signal(data["timestamp"].iloc[-1], self.name, "buy", float(-z))
        return None


class MomentumModel(StrategyModel):
    def __init__(self, lookback: int = 10) -> None:
        super().__init__(name="Momentum", warmup=lookback)
        self.lookback = lookback

    def update(self, data: pd.DataFrame) -> Optional[Signal]:
        prices = data["price"].to_numpy()
        if len(prices) < self.lookback + 1:
            return None

        change = prices[-1] - prices[-1 - self.lookback]
        if change > 0:
            return Signal(data["timestamp"].iloc[-1], self.name, "buy", float(change))
        if change < 0:
            return Signal(data["timestamp"].iloc[-1], self.name, "sell", float(-change))
        return None

In [ ]:
class SimpleKalmanFilterModel(StrategyModel):
    def __init__(self) -> None:
        super().__init__(name="KalmanFilter", warmup=2)
        self.estimate = None
        self.uncertainty = 1.0
        self.process_variance = 1e-3
        self.measurement_variance = 1e-1

    def update(self, data: pd.DataFrame) -> Optional[Signal]:
        price = float(data["price"].iloc[-1])
        if self.estimate is None:
            self.estimate = price
            return None

        self.uncertainty += self.process_variance
        kalman_gain = self.uncertainty / (self.uncertainty + self.measurement_variance)
        self.estimate += kalman_gain * (price - self.estimate)
        self.uncertainty *= 1 - kalman_gain

        error = price - self.estimate
        if abs(error) > 0.5:
            return Signal(data["timestamp"].iloc[-1], self.name, "buy" if error < 0 else "sell", float(abs(error)))
        return None


class ARIMAModel(StrategyModel):
    def __init__(self, p: int = 1, d: int = 1, q: int = 0) -> None:
        super().__init__(name="ARIMA", warmup=10)
        self.order = (p, d, q)

    def update(self, data: pd.DataFrame) -> Optional[Signal]:
        try:
            from statsmodels.tsa.arima.model import ARIMA
        except ImportError:
            return None

        prices = data["price"].to_numpy()
        if len(prices) < max(self.order) + 5:
            return None

        model = ARIMA(prices, order=self.order)
        fitted = model.fit(method="innovations_mle")
        forecast = fitted.forecast(steps=1)
        if len(forecast) == 0:
            return None

        next_price = float(forecast[0])
        diff = next_price - prices[-1]
        if diff > 0:
            return Signal(data["timestamp"].iloc[-1], self.name, "buy", float(diff))
        if diff < 0:
            return Signal(data["timestamp"].iloc[-1], self.name, "sell", float(-diff))
        return None


class TradingAgent:
    def __init__(self, models: List[StrategyModel]) -> None:
        self.models: Dict[str, StrategyModel] = {model.name: model for model in models}
        self.history: pd.DataFrame = pd.DataFrame(columns=["timestamp", "price"])

    def update_price(self, timestamp: pd.Timestamp, price: float) -> List[Signal]:
        self.history = pd.concat(
            [self.history, pd.DataFrame([{"timestamp": timestamp, "price": price}])],
            ignore_index=True,
        )
        signals: List[Signal] = []
        for model in self.models.values():
            signal = model.update(self.history)
            if signal is not None:
                signals.append(signal)
        return signals

    def run_backtest(self, data: pd.DataFrame) -> List[Signal]:
        signals: List[Signal] = []
        self.history = pd.DataFrame(columns=["timestamp", "price"])
        for _, row in data.iterrows():
            signals.extend(self.update_price(row["timestamp"], float(row["price"])))
        return signals

In [25]:
root = Path(".").resolve()
data_path = root / "data" / "SP500.csv"
print("Loading data from:", data_path)

symbol = "AAPL UW Equity"

try:
    df = DataLoader.load_csv(str(data_path), price_column=symbol)
except FileNotFoundError:
    raise
except Exception as exc:
    raise RuntimeError(f"Unable to load data for {symbol}: {exc}")

print("Data loaded:", df.shape)
df.head()

Loading data from: /Users/mustapha/Mon_disque/Learning/Quant with Python/data/SP500.csv
Data loaded: (426, 2)


,timestamp,price
0,2016-01-04,105.35
1,2016-01-05,102.71
2,2016-01-06,100.70
3,2016-01-07,96.45
4,2016-01-08,96.96


In [26]:
agent = TradingAgent(
    models=[
        MovingAverageCrossoverModel(fast_window=8, slow_window=21),
        RSIModel(window=14),
        MeanReversionModel(window=20, threshold=2.0),
        MomentumModel(lookback=10),
        SimpleKalmanFilterModel(),
        ARIMAModel(p=1, d=1, q=0),
    ]
)

signals = agent.run_backtest(df)
print("Generated signals:", len(signals))

if signals:
    signals_df = pd.DataFrame([
        {
            "timestamp": signal.timestamp,
            "model": signal.model_name,
            "signal": signal.signal,
            "score": signal.score,
        }
        for signal in signals
    ])
    display(signals_df.head(30))
    display(signals_df["model"].value_counts())
else:
    print("No signals were generated.")

/var/folders/fx/39y8j8751j37c9vmjzj_2ftm0000gn/T/ipykernel_4111/402185572.py:62: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  self.history = pd.concat(
/var/folders/fx/39y8j8751j37c9vmjzj_2ftm0000gn/T/ipykernel_4111/402185572.py:42: UserWarning: Provided `endog` series has been differenced to eliminate integration prior to parameter estimation by method "innovations_mle".
  fitted = model.fit(method="innovations_mle")
/var/folders/fx/39y8j8751j37c9vmjzj_2ftm0000gn/T/ipykernel_4111/402185572.py:42: UserWarning: Provided `endog` series has been differenced to eliminate integration prior to parameter estimation by method "innovations_mle".
  fitted = model.fit(method="innovations_mle")
/var/folders/fx/39y8j8751j37c9vmjzj_2ftm0000gn/T/ipykerne

Generated signals: 1398


/var/folders/fx/39y8j8751j37c9vmjzj_2ftm0000gn/T/ipykernel_4111/402185572.py:42: UserWarning: Provided `endog` series has been differenced to eliminate integration prior to parameter estimation by method "innovations_mle".
  fitted = model.fit(method="innovations_mle")


,timestamp,model,signal,score
0,2016-01-06,KalmanFilter,buy,1.172266
1,2016-01-07,KalmanFilter,buy,3.641690
2,2016-01-08,KalmanFilter,buy,2.339907
3,2016-01-11,KalmanFilter,buy,0.609668
4,2016-01-11,ARIMA,buy,0.685671
5,2016-01-12,KalmanFilter,sell,0.673437
6,2016-01-12,ARIMA,buy,0.693614
7,2016-01-13,KalmanFilter,buy,1.595000
8,2016-01-13,ARIMA,sell,0.868979
9,2016-01-14,ARIMA,buy,0.303859


model
ARIMA                     420
Momentum                  415
KalmanFilter              394
RSI                        90
MeanReversion              68
MovingAverageCrossover     11
Name: count, dtype: int64

In [ ]:
def MSE(y_1, y_2):
    